# Result · R0 reproduces the published SSL-MOS baseline

Our undefended predictor on BVCC test versus the VMC'22 B01 SSL-MOS reference (utterance SRCC ≈ 0.87, system SRCC
≈ 0.92). Recomputes the E2 evaluation of the three R0 seeds (~3 min each).

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters (also settable through environment variables for non-interactive runs)
# SOURCE = "recompute": re-run the evaluations behind this output from the released checkpoints (GPU), then aggregate.
# SOURCE = "released" : aggregate the released outputs of the original runs (CPU, seconds). This rewrites the
#                       committed analysis/out/ and paper/ files, which should come out identical.
SOURCE = os.environ.get("NB_SOURCE", "recompute")
# RUN = "slurm": one GPU job per run (sbatch), this kernel waits; "local": run in this kernel (needs a GPU).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['e2'] and v.get("predictor") == "R0" or k.startswith("e2_R0_")]
print(len(KEYS), "runs:", ", ".join(KEYS))

3 runs: e2_R0_s0, e2_R0_s1, e2_R0_s2


In [4]:
if SOURCE == "recompute":
    repro.run_keys(KEYS, "recompute", RUN)  # skips runs whose recomputed output already exists
dirs = repro.run_analysis(SOURCE)  # whole analysis chain (seconds); falls back to released outputs where needed
OUT, PAPER = dirs["out"], dirs["paper"]
COMMITTED_OUT, COMMITTED_PAPER = ROOT / "analysis" / "out", ROOT / "paper"

[analysis] runs=/fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/runs
[analysis] out=/users/PAS2301/alialavi/projects/audio_assessment_hackability/analysis/out
[analysis] paper=/users/PAS2301/alialavi/projects/audio_assessment_hackability/paper


In [5]:
# Where this notebook's inputs came from: recomputed from checkpoints, or released outputs of the original runs.
used = repro.load_registry(repro.results_registry(SOURCE))
pd.Series({k: used[k].get("source", "released") for k in KEYS}, name="source").value_counts()

source
released    3
Name: count, dtype: int64

In [6]:
display(Markdown((OUT / "r0_fidelity.md").read_text()))

VMC'22 B01 SSL-MOS reference (BVCC test): utt SRCC ≈ 0.87, sys SRCC ≈ 0.92 (tolerance ±0.03)

|   seed |   utt_mse |   utt_srcc |   sys_mse |   sys_srcc |
|-------:|----------:|-----------:|----------:|-----------:|
|  0.000 |     0.230 |      0.877 |     0.108 |      0.922 |
|  1.000 |     0.226 |      0.873 |     0.108 |      0.918 |
|  2.000 |     0.247 |      0.883 |     0.124 |      0.926 |

In [7]:
# Numbers the paper quotes from this output: committed value vs this notebook's value.
cmp = repro.compare_numbers(OUT / "paper_numbers.json", prefixes=('r0_bvcc',))
print(f"{len(cmp)} numbers; max |diff| = {cmp.abs_diff.max():.4f}; identical: {(cmp.abs_diff == 0).sum()}")
cmp.head(20)

4 numbers; max |diff| = 0.0000; identical: 4


,paper,reproduced,abs_diff
r0_bvcc_sys_srcc,0.922,0.922,0.0
r0_bvcc_sys_srcc_sd,0.004,0.004,0.0
r0_bvcc_utt_srcc,0.878,0.878,0.0
r0_bvcc_utt_srcc_sd,0.005,0.005,0.0
